# Fig 1: dependence of the clustering on k

This notebook draws Fig 1 of the manuscript: KMeans partitions of the Levine mass cytometry data at k = 6, 10, 14 and 18, and of the Klein scRNA-seq data at k = 2, 3, 4 and 5.

Every panel is a plausible-looking partition, and the scatter plots alone do not say which k describes the data. On these data each k would imply a different number of cell types or cell stages. The Klein and Levine notebooks show CARVE's selections on the same data.

## Running this notebook

- Install the package with the benchmarks and notebooks extras: `pip install -e ".[benchmarks,notebooks]"` from the repository root.
- The Levine data are read through rpy2 from HDCytoData (`make levine-setup`); the Klein files go in `data/Klein/`. The Levine and Klein notebooks describe both data sets.
- The figure is written to `vis/motivational_plots/clustering_problem.png` and `vis/plos/Fig1.tif`. Copying it into the manuscript is a manual step.

## 1. Setup

Imports, a display helper, the seed, and a function that relabels one partition to match another so that matching clusters get the same color.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from benchmarks.datasets import load_klein, load_levine32
from benchmarks.figures import DatasetRow, figure_clustering_problem

In [ ]:
def show(fig):
    """Display a figure once, then close it so the inline backend does not draw it again."""
    display(fig)
    plt.close(fig)

In [ ]:
RANDOM_SEED = 42


In [7]:
def align_cluster_labels(
    reference_labels: np.ndarray,
    labels: np.ndarray,
) -> np.ndarray:
    """Relabel `labels` to best match `reference_labels`.

    The matching is a Hungarian assignment on the contingency table of the two
    partitions, over the union of their label values.
    """
    all_classes = np.union1d(reference_labels, labels)
    cont = np.zeros((len(all_classes), len(all_classes)), dtype=int)

    ref_to_idx = {val: i for i, val in enumerate(all_classes)}
    lab_to_idx = {val: i for i, val in enumerate(all_classes)}

    for r, l in zip(reference_labels, labels):
        cont[ref_to_idx[r], lab_to_idx[l]] += 1

    row_ind, col_ind = linear_sum_assignment(-cont)
    mapping = {all_classes[c]: all_classes[r] for r, c in zip(row_ind, col_ind)}

    aligned = np.array([mapping[lbl] for lbl in labels], dtype=reference_labels.dtype)
    return aligned

In [9]:
%env PYTHONWARNINGS=ignore::FutureWarning,ignore::DeprecationWarning,ignore::UserWarning,ignore::RuntimeWarning

env: PYTHONWARNINGS=ignore::FutureWarning,ignore::DeprecationWarning,ignore::UserWarning,ignore::RuntimeWarning


## 2. Data

Levine: a stratified subsample of 5,000 gated cells. Klein: all 2,717 cells.

In [ ]:
# The manuscript's stratified subsample of 5,000 gated cells.
X_levine, y_levine, meta_levine = load_levine32(subsample=5000)

In [ ]:
X_klein, y_klein, meta_klein = load_klein(root=Path("../../data"))

## 3. KMeans partitions

KMeans at each k, with a fixed seed.

In [53]:
levine_clusters_6 = KMeans(n_clusters=6, random_state=RANDOM_SEED).fit_predict(X_levine)
levine_clusters_10 = KMeans(n_clusters=10, random_state=RANDOM_SEED).fit_predict(
    X_levine
)
levine_clusters_14 = KMeans(n_clusters=14, random_state=RANDOM_SEED).fit_predict(
    X_levine
)
levine_clusters_18 = KMeans(n_clusters=18, random_state=RANDOM_SEED).fit_predict(
    X_levine
)

klein_clusters_2 = KMeans(n_clusters=2, random_state=RANDOM_SEED).fit_predict(X_klein)
klein_clusters_3 = KMeans(n_clusters=3, random_state=RANDOM_SEED).fit_predict(X_klein)
klein_clusters_4 = KMeans(n_clusters=4, random_state=RANDOM_SEED).fit_predict(X_klein)
klein_clusters_5 = KMeans(n_clusters=5, random_state=RANDOM_SEED).fit_predict(X_klein)

KMeans numbers its clusters arbitrarily. Each partition is relabeled to match the middle one, 14 clusters for Levine and 4 for Klein, so that a cluster keeps its color across panels. The Klein labels are then permuted by hand (label_map) to set the colors in the figure. The two `np.unique` cells show which labels the Klein partitions use.

In [56]:
levine_clusters_6 = align_cluster_labels(levine_clusters_14, levine_clusters_6)
levine_clusters_10 = align_cluster_labels(levine_clusters_14, levine_clusters_10)
levine_clusters_18 = align_cluster_labels(levine_clusters_14, levine_clusters_18)

klein_clusters_2 = align_cluster_labels(klein_clusters_4, klein_clusters_2)
klein_clusters_3 = align_cluster_labels(klein_clusters_4, klein_clusters_3)
klein_clusters_5 = align_cluster_labels(klein_clusters_4, klein_clusters_5)

In [57]:
np.unique(klein_clusters_4)

array([0, 1, 2, 3], dtype=int32)

In [58]:
label_map = {0: 3, 1: 2, 2: 0, 3: 1, 4: 4}

klein_clusters_2 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_2])
klein_clusters_3 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_3])
klein_clusters_4 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_4])
klein_clusters_5 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_5])

In [59]:
np.unique(klein_clusters_3)

array([0, 1, 3])

## 4. Figure

A t-SNE of the Levine cells and the first two principal components of the Klein cells, each drawn with the partitions above (Fig 1).

In [ ]:
levine_embedding = TSNE(random_state=RANDOM_SEED).fit_transform(X_levine)
klein_embedding = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X_klein)

# Marker areas are for a 16-inch-wide layout; the figure scales them to print width.
rows = [
    DatasetRow(
        name="Levine",
        embedding=levine_embedding,
        partitions={
            6: levine_clusters_6,
            10: levine_clusters_10,
            14: levine_clusters_14,
            18: levine_clusters_18,
        },
        marker_size=12,
    ),
    DatasetRow(
        name="Klein",
        embedding=klein_embedding,
        partitions={
            2: klein_clusters_2,
            3: klein_clusters_3,
            4: klein_clusters_4,
            5: klein_clusters_5,
        },
        marker_size=30,
    ),
]
show(figure_clustering_problem(rows))